# Floor Plan Generation Model - Google Colab
Training a conditional U-Net to generate floor plans from bounding box specifications.

This notebook uses free GPU/TPU resources from Google Colab.

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision pillow scipy -q
print("Dependencies installed!")
!unzip -q /content/data-new.zip -d /content/data-new
print("Data unzip'd")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

## 3. Check GPU/TPU

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 4. Import Libraries and Define Model

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F
from scipy import ndimage

# Room color mapping
ROOM_COLORS = {
    (0, 0, 0): 0,  # Walls/Openings
    (230, 230, 230): 1,  # Hall
    (200, 230, 230): 2,  # Corridor
    (230, 200, 230): 3,  # Living Room
    (230, 230, 200): 4,  # Kitchen
    (230, 200, 200): 5,  # Bathroom
    (200, 200, 230): 6,  # Single Bedroom
    (170, 170, 230): 7,  # Master Bedroom
    (230, 170, 170): 8,  # Private Bathroom
    (140, 140, 230): 9,  # Double Bedroom A
    (110, 110, 230): 10,  # Double Bedroom B
    (230, 170, 230): 11,  # Living Room with Kitchenette
}

print("Libraries imported successfully!")

## 5. Define Dataset Class

In [ ]:
class FloorPlanDataset(Dataset):
    """Dataset for floor plan images."""
    
    def __init__(self, data_dir, image_size=128, augment=True, limit=None):
        """
        Args:
            data_dir: Root directory containing subdirectories with floor plans
            image_size: Target size to resize images
            augment: Whether to apply data augmentation
            limit: Limit number of images (useful for testing)
        """
        self.image_size = image_size
        self.augment = augment
        
        # Find all PNG files recursively
        self.image_paths = []
        for root, dirs, files in os.walk(data_dir):
            for file in files:
                if file.endswith('.png'):
                    self.image_paths.append(os.path.join(root, file))
        
        # Limit if specified
        if limit:
            self.image_paths = self.image_paths[:limit]
        
        print(f"Found {len(self.image_paths)} floor plan images")
        
        # Image transforms
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def rgb_to_class_map(self, img_array):
        """Convert RGB floor plan to class segmentation map."""
        h, w, c = img_array.shape
        class_map = np.zeros((h, w), dtype=np.int64)
        
        for color, class_id in ROOM_COLORS.items():
            mask = np.all(np.abs(img_array - np.array(color).reshape(1, 1, 3)) <= 5, axis=2)
            class_map[mask] = class_id
        
        return class_map
    
    def extract_bbox_features(self, img_array):
        """Extract bounding box features from floor plan."""
        non_wall = np.any(img_array > 10, axis=2)
        
        if not non_wall.any():
            return np.array([1.0, 1.0, 1.0, 1.0])
        
        rows = np.any(non_wall, axis=1)
        cols = np.any(non_wall, axis=0)
        
        y_min, y_max = np.where(rows)[0][[0, -1]]
        x_min, x_max = np.where(cols)[0][[0, -1]]
        
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        aspect_ratio = width / height
        area = non_wall.sum() / (img_array.shape[0] * img_array.shape[1])
        
        return np.array([
            width / img_array.shape[1],
            height / img_array.shape[0],
            aspect_ratio,
            area
        ], dtype=np.float32)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        
        img_array = np.array(img)
        bbox_features = self.extract_bbox_features(img_array)
        class_map = self.rgb_to_class_map(img_array)
        
        class_map_pil = Image.fromarray(class_map.astype(np.uint8))
        class_map_resized = class_map_pil.resize((self.image_size, self.image_size), Image.NEAREST)
        class_map_tensor = torch.from_numpy(np.array(class_map_resized)).long()
        
        img_tensor = self.transform(img)
        
        return {
            'bbox_features': torch.from_numpy(bbox_features),
            'class_map': class_map_tensor,
            'rgb_image': img_tensor,
            'path': img_path
        }

print("FloorPlanDataset defined!")

## 6. Define Model Architecture

In [ ]:
class ConditionalUNet(nn.Module):
    """Conditional U-Net for floor plan generation."""
    
    def __init__(self, condition_dim=4, num_classes=12, base_channels=32):
        super(ConditionalUNet, self).__init__()
        
        self.condition_dim = condition_dim
        self.num_classes = num_classes
        
        # Condition embedding network
        self.condition_embed = nn.Sequential(
            nn.Linear(condition_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
        )
        
        # Encoder
        self.enc1 = self.conv_block(3, base_channels)
        self.enc2 = self.conv_block(base_channels, base_channels * 2)
        self.enc3 = self.conv_block(base_channels * 2, base_channels * 4)
        self.enc4 = self.conv_block(base_channels * 4, base_channels * 8)
        
        # Bottleneck
        self.bottleneck = self.conv_block(base_channels * 8 + 128, base_channels * 8)
        
        # Decoder
        self.dec4 = self.conv_block(base_channels * 16, base_channels * 4)
        self.dec3 = self.conv_block(base_channels * 8, base_channels * 2)
        self.dec2 = self.conv_block(base_channels * 4, base_channels)
        self.dec1 = self.conv_block(base_channels * 2, base_channels)
        
        # Output
        self.out_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)
        self.pool = nn.MaxPool2d(2)
        
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x, condition):
        # Encode
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        
        # Pool and embed condition
        pooled_enc4 = self.pool(enc4)
        cond_embed = self.condition_embed(condition)
        cond_embed = cond_embed.view(cond_embed.size(0), -1, 1, 1)
        cond_embed = cond_embed.expand(-1, -1, pooled_enc4.size(2), pooled_enc4.size(3))
        
        # Bottleneck
        bottleneck = torch.cat([pooled_enc4, cond_embed], dim=1)
        bottleneck = self.bottleneck(bottleneck)
        
        # Decode
        dec4 = F.interpolate(bottleneck, scale_factor=2, mode='bilinear', align_corners=False)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.dec4(dec4)
        
        dec3 = F.interpolate(dec4, scale_factor=2, mode='bilinear', align_corners=False)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = F.interpolate(dec3, scale_factor=2, mode='bilinear', align_corners=False)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = F.interpolate(dec2, scale_factor=2, mode='bilinear', align_corners=False)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)
        
        out = self.out_conv(dec1)
        return out

print("ConditionalUNet model defined!")

## 7. Training Function

In [ ]:
def train_model(data_dir, output_dir, epochs=10, batch_size=16, lr=0.0001, image_size=128, limit_images=None):
    """Train the floor plan generation model."""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Dataset
    dataset = FloorPlanDataset(data_dir, image_size=image_size, augment=True, limit=limit_images)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Model
    model = ConditionalUNet(condition_dim=4, num_classes=12, base_channels=32).to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    # Training loop
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for batch_idx, batch in enumerate(train_loader):
            rgb_images = batch['rgb_image'].to(device)
            bbox_features = batch['bbox_features'].to(device)
            class_maps = batch['class_map'].to(device)
            
            # Forward
            outputs = model(rgb_images, bbox_features)
            loss = criterion(outputs, class_maps)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            if batch_idx % max(1, len(train_loader)//5) == 0:
                print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                rgb_images = batch['rgb_image'].to(device)
                bbox_features = batch['bbox_features'].to(device)
                class_maps = batch['class_map'].to(device)
                
                outputs = model(rgb_images, bbox_features)
                loss = criterion(outputs, class_maps)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        print(f"\nEpoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}\n")
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, os.path.join(output_dir, 'best_model.pth'))
            print(f"Saved best model with val_loss: {val_loss:.4f}")
    
    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Progress')
    plt.savefig(os.path.join(output_dir, 'training_loss.png'))
    plt.show()
    
    print("Training complete!")
    return model

print("Training function defined!")

## 8. Set Up Paths and Train

In [ ]:
# Update these paths based on your Google Drive structure
data_dir = "/content/drive/MyDrive/lotiv-ai/data-new"  # Adjust if needed
output_dir = "/content/drive/MyDrive/lotiv-ai/models/floor_plan_generator"

# Check if data directory exists
if os.path.exists(data_dir):
    print(f"✓ Data directory found: {data_dir}")
    subdirs = os.listdir(data_dir)[:3]
    print(f"Sample subdirectories: {subdirs}")
else:
    print(f"✗ Data directory not found: {data_dir}")
    print("Please update the data_dir path in the cell above")

In [ ]:
# Start training
print("Starting training...")
model = train_model(
    data_dir=data_dir,
    output_dir=output_dir,
    epochs=10,
    batch_size=16,
    lr=0.0001,
    image_size=128,
    limit_images=None  # Set to a number like 500 to limit dataset for faster testing
)

## 9. Generate Floor Plans (Optional)

In [ ]:
def generate_floor_plan(model_path, bbox_features, device='cuda'):
    """Generate a floor plan given bounding box features."""
    if not os.path.exists(model_path):
        print(f"Model not found at {model_path}")
        return
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model
    model = ConditionalUNet(condition_dim=4, num_classes=12, base_channels=32).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Prepare input
    bbox_tensor = torch.from_numpy(np.array(bbox_features, dtype=np.float32)).unsqueeze(0).to(device)
    noise = torch.randn(1, 3, 128, 128).to(device)
    
    # Generate
    with torch.no_grad():
        output = model(noise, bbox_tensor)
        predicted_classes = torch.argmax(output, dim=1).cpu().numpy()[0]
    
    # Convert to RGB
    color_map = {v: k for k, v in ROOM_COLORS.items()}
    rgb_image = np.zeros((128, 128, 3), dtype=np.uint8)
    
    for class_id, color in color_map.items():
        mask = predicted_classes == class_id
        rgb_image[mask] = color
    
    return rgb_image

# Example: Generate a floor plan
model_path = os.path.join(output_dir, 'best_model.pth')
bbox_features = [0.8, 0.6, 1.33, 0.48]  # width, height, aspect_ratio, area

if os.path.exists(model_path):
    generated_plan = generate_floor_plan(model_path, bbox_features)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(generated_plan)
    plt.title(f"Generated Floor Plan (bbox: {bbox_features})")
    plt.axis('off')
    plt.savefig(os.path.join(output_dir, 'generated_example.png'))
    plt.show()
else:
    print(f"Model not found. Please train first.")

## Notes

- **Update data_dir path** in cell 8 to match your Google Drive folder structure
- First run may be slow due to data loading
- Check **Runtime → Change runtime type** to enable GPU (or TPU)
- Models are saved to Google Drive automatically
- Adjust `limit_images` parameter to test quickly on a subset